# **05_colombia_cleaning**

## GOAL
- The data that we have in Neon is raw, so here we're going to clean it and load it into the processed schema in Neon.
- The idea is to clean the data to answer this question: What is the state of credit delinquency in Colombia today, how has it changed since 2015, and in which types of credit and institutions is it concentrated?
- We're going to bring the data, look at it, clean it, check it and save it in processed.
- At the end, we'll have a clean table with one row per entity, month and product, with the calculated delinquency rate.
- The next step is to move the cleaning process to src/ so it runs in GitHub Actions.

## Part 1 – Bring the raw data

In [1]:
import sys
sys.path.append("..")
import pandas as pd
from src.load import get_engine

In [2]:
pd.set_option("display.float_format", "{:,.2f}".format)
pd.set_option("display.max_columns", None)

In [3]:
engine = get_engine()

In [4]:
df_raw = pd.read_sql("SELECT * FROM raw.co_loan_portfolio", engine) # Bring the raw data from Neon

In [5]:
df_raw.shape # verify how many rows and columns the dataset has

(113418, 34)

In [6]:
df_raw.dtypes # Check the data type of each column

tipo_entidad                   str
codigo_entidad                 str
nombreentidad                  str
fecha_corte                    str
unicap                         str
descrip_uc                     str
renglon                        str
desc_renglon                   str
_1_saldo_de_la_cartera_a       str
_2_vigente                     str
_3_vencida_1_2_meses           str
_4_vencida_2_3_meses           str
_5_vencida_1_3_meses           str
_6_vencida_3_4_meses           str
_7_vencida_de_4_meses          str
_8_vencida_3_6_meses           str
_9_vencida_6_meses             str
_10_vencida_1_4_meses          str
_11_vencida_4_6_meses          str
_12_vencida_6_12_meses         str
_13_vencida_12_18_meses        str
_14_vencida_12_meses           str
_15_vencida_18_meses           str
_16_n_mero_de_clientes_mora    str
_17_calificaci_n_de_riesgo     str
_18_calificaci_n_de_riesgo     str
_19_calificaci_n_de_riesgo     str
_20_calificaci_n_de_riesgo     str
_21_calificaci_n_de_

## Part 2 – Look at the data

In [7]:
df_raw.head(3)

,tipo_entidad,codigo_entidad,nombreentidad,fecha_corte,unicap,descrip_uc,renglon,desc_renglon,_1_saldo_de_la_cartera_a,_2_vigente,_3_vencida_1_2_meses,_4_vencida_2_3_meses,_5_vencida_1_3_meses,_6_vencida_3_4_meses,_7_vencida_de_4_meses,_8_vencida_3_6_meses,_9_vencida_6_meses,_10_vencida_1_4_meses,_11_vencida_4_6_meses,_12_vencida_6_12_meses,_13_vencida_12_18_meses,_14_vencida_12_meses,_15_vencida_18_meses,_16_n_mero_de_clientes_mora,_17_calificaci_n_de_riesgo,_18_calificaci_n_de_riesgo,_19_calificaci_n_de_riesgo,_20_calificaci_n_de_riesgo,_21_calificaci_n_de_riesgo,_22_calificaci_n_de_riesgo,_23_calificaci_n_de_riesgo,_24_calificaci_n_de_riesgo,_25_calificaci_n_de_riesgo,_26_calificaci_n_de_riesgo
0,1,12,Banco Gnb Sudameris,2015-01-31T00:00:00.000,1,CRÉDITO ROTATIVO,5,CRÉDITO ROTATIVO TOTAL,37651351298,34981440061,768061743,579918756,0,0,0,1077855545,244075193,0,0,0,0,0,0,469,6176,32878868825.55,270,1835206170,142,975343138,326,1368695340,69,593093538
1,1,6,Itaú Banco Itaú.,2015-01-31T00:00:00.000,1,CRÉDITO ROTATIVO,5,CRÉDITO ROTATIVO TOTAL,476766070943.32,453506953108.61,8052110662.22,3950013841.33,0,0,0,6973931783.54,4283061547.62,0,0,0,0,0,0,2476,43553,442277573569.18,1356,13043155083.47,951,7559380077.26,965,8135341560.27,671,5750620653.14
2,1,9,Citibank,2015-01-31T00:00:00.000,1,CRÉDITO ROTATIVO,5,CRÉDITO ROTATIVO TOTAL,826852228376.59,782016598063.64,9271463390.82,8065650991.89,0,0,0,17024347599.27,10474168330.97,0,0,0,0,0,0,5830,125128,772666404396.07,1840,11740703688.96,1486,9872854504.87,1854,14111818486.48,2469,18460447300.21


In [8]:
print(df_raw.isna().sum().sum())    # Check for any NULL value or empty value
print((df_raw == "").sum().sum())

0
0


In [9]:
df_raw["descrip_uc"].nunique()

32

In [10]:
df_raw["descrip_uc"].value_counts()

descrip_uc
TARJETAS DE CRÉDITO                        11231
CARTERA COMERCIAL LEASING FINANCIERO        9354
LIBRANZA                                    9117
CARTERA COMERCIAL CONSTRUCCION              8068
CARTERA COMERCIAL MICROEMPRESA              5308
LIBRE INVERSIÓN                             5171
CARTERA COMERCIAL PYMES                     5064
CARTERA COMERCIAL EMPRESARIAL               4903
CARTERA COMERCIAL CORPORATIVO               4615
CREDITOS DE CONSUMO PARA EMPLEADOS          4118
VEHÍCULO                                    3470
VIVIENDA NO VIS PESOS                       3432
OTROS PORTAFOLIOS DE CONSUMO                3398
MICROCREDITOS < O IGUALES A 25 SMMLV        3207
VIVIENDA VIS PESOS                          2985
MICROCREDITOS > 25  Y HASTA 120 SMMLV       2966
CREDITOS DE VIVIENDA PARA EMPL NO VIS       2928
CARTERA COMERCIAL FINAN E INSTITUCIO        2770
CRÉDITO ROTATIVO                            2750
CARTERA COMERCIAL OFICIAL O GOBIERNO        2533
CREDITOS 

In [11]:
df_raw["desc_renglon"].nunique()

57

In [12]:
df_raw["desc_renglon"].value_counts()

desc_renglon
MICROEMPRESA                                5308
LIBRE INVERSIÓN                             5171
PYMES                                       5064
EMPRESARIAL                                 4903
CORPORATIVO                                 4615
LIBRANZA TOTAL                              4274
LIBRANZA OTROS                              4273
VEHÍCULO                                    3470
TARJETA DE CRÉDITO TOTAL                    3459
NO VIS PESOS TOTAL                          3432
CONSUMO OTROS TOTAL                         3398
TARJETA DE CRÉDITO INGRESO HASTA 2 SMMLV    3304
CRÉDITOS DE CONSUMO PARA EMPLEADOS          3299
MICRO MENOR TOTAL                           3207
VIS PESOS TOTAL                             2985
MICRO MAYOR TOTAL                           2966
FINANCIERO E INSTITUCIONAL                  2770
CRÉDITO ROTATIVO TOTAL                      2750
TARJETA CREDITO INGRESO MAYOR 2SMMLV        2677
CRÉDITO EMPLEADOS VIVIENDA NO VIS           2640
CONSTRU

### FINDINGS

- The dataset has no NULL values or empty values
- We've found that `descrip_uc` is a group, and inside each group there are one or more products described in `desc_renglon`
- There are 32 groups and 57 products.
- All columns are stored as text, so we need to convert the numbers and the date.
- `fecha_corte` includes a time (`00:00:00`) that we don’t need. We’ll keep only the date.
- 19 products are subtotals (they end in TOTAL). Some groups only have the TOTAL row, so we can’t delete all of them. Rule: keep one row per entity, month and group: the TOTAL row if the group has one, or the only row if it doesn’t.
- 32 groups are too many for a chart, so we’ll put them into 4 credit types: commercial, consumer, housing and microcredit.
- Product names in `desc_renglon` have typos and abbreviations (for example `PROYRCTOS`, `LEASIN`), so we’ll use the group names in `descrip_uc` instead.
- Column names like `_1_saldo_de_la_cartera_a` are hard to read. We’ll rename them in English.

## Part 3 – Clean the data

### 3.1 Rename and select columns

- We kept only 21 out of 34 columns because those are the ones we need for the analysis.
- We dropped columns from _3 to _15 because we don't need them. They split the past-due balance by how many months the portfolio has been unpaid, and each credit type uses different ranges, so most of them are 0. We get the total past-due balance by subtracting `current_balance` from `total_balance`. The original columns are still in `raw.co_loan_portfolio` if we need them later.
- We renamed the columns to English so the code is easier to read.

In [13]:
COLUMN_NAMES = {
    "tipo_entidad": "entity_type",
    "codigo_entidad": "entity_code",
    "nombreentidad": "entity_name",
    "fecha_corte": "cutoff_date",
    "unicap": "group_code",
    "descrip_uc": "group_name",
    "renglon": "product_code",
    "desc_renglon": "product_name",
    "_1_saldo_de_la_cartera_a": "total_balance",
    "_2_vigente": "current_balance",
    "_16_n_mero_de_clientes_mora": "clients_past_due",
    "_17_calificaci_n_de_riesgo": "risk_a_clients",
    "_18_calificaci_n_de_riesgo": "risk_a_balance",
    "_19_calificaci_n_de_riesgo": "risk_b_clients",
    "_20_calificaci_n_de_riesgo": "risk_b_balance",
    "_21_calificaci_n_de_riesgo": "risk_c_clients",
    "_22_calificaci_n_de_riesgo": "risk_c_balance",
    "_23_calificaci_n_de_riesgo": "risk_d_clients",
    "_24_calificaci_n_de_riesgo": "risk_d_balance",
    "_25_calificaci_n_de_riesgo": "risk_e_clients",
    "_26_calificaci_n_de_riesgo": "risk_e_balance"
}

In [14]:
df_clean = df_raw[list(COLUMN_NAMES)].rename(columns=COLUMN_NAMES)

In [15]:
df_clean.shape

(113418, 21)

In [16]:
df_clean.head(3)

,entity_type,entity_code,entity_name,cutoff_date,group_code,group_name,product_code,product_name,total_balance,current_balance,clients_past_due,risk_a_clients,risk_a_balance,risk_b_clients,risk_b_balance,risk_c_clients,risk_c_balance,risk_d_clients,risk_d_balance,risk_e_clients,risk_e_balance
0,1,12,Banco Gnb Sudameris,2015-01-31T00:00:00.000,1,CRÉDITO ROTATIVO,5,CRÉDITO ROTATIVO TOTAL,37651351298,34981440061,469,6176,32878868825.55,270,1835206170,142,975343138,326,1368695340,69,593093538
1,1,6,Itaú Banco Itaú.,2015-01-31T00:00:00.000,1,CRÉDITO ROTATIVO,5,CRÉDITO ROTATIVO TOTAL,476766070943.32,453506953108.61,2476,43553,442277573569.18,1356,13043155083.47,951,7559380077.26,965,8135341560.27,671,5750620653.14
2,1,9,Citibank,2015-01-31T00:00:00.000,1,CRÉDITO ROTATIVO,5,CRÉDITO ROTATIVO TOTAL,826852228376.59,782016598063.64,5830,125128,772666404396.07,1840,11740703688.96,1486,9872854504.87,1854,14111818486.48,2469,18460447300.21


### 3.2 Convert data types

- Every column arrived as text, so we had to change the data types before doing any calculations.
- We split the number columns into two groups: balances became `float` because money can have decimals, and clients became `int` because people are counted as whole numbers. We also turned `cutoff_date` into `datetime` so we can sort the data and build the timeline.
- We kept the rest as text because they are codes and names: they identify things, and we never do math with them.
- We checked the result with `.info()`: every column has the expected type, and all 112,602 values are still non-null.

In [17]:
BALANCE_COLUMNS = ["total_balance", "current_balance", "risk_a_balance", "risk_b_balance", "risk_c_balance", "risk_d_balance", "risk_e_balance"]
CLIENT_COLUMNS = ["clients_past_due", "risk_a_clients", "risk_b_clients", "risk_c_clients", "risk_d_clients", "risk_e_clients"]

In [18]:
df_clean[BALANCE_COLUMNS] = df_clean[BALANCE_COLUMNS].astype(float)
df_clean[CLIENT_COLUMNS] = df_clean[CLIENT_COLUMNS].astype(int)
df_clean["cutoff_date"] = pd.to_datetime(df_clean["cutoff_date"])

In [19]:
df_clean.info() # check the new data types

<class 'pandas.DataFrame'>
RangeIndex: 113418 entries, 0 to 113417
Data columns (total 21 columns):
 #   Column            Non-Null Count   Dtype         
---  ------            --------------   -----         
 0   entity_type       113418 non-null  str           
 1   entity_code       113418 non-null  str           
 2   entity_name       113418 non-null  str           
 3   cutoff_date       113418 non-null  datetime64[us]
 4   group_code        113418 non-null  str           
 5   group_name        113418 non-null  str           
 6   product_code      113418 non-null  str           
 7   product_name      113418 non-null  str           
 8   total_balance     113418 non-null  float64       
 9   current_balance   113418 non-null  float64       
 10  clients_past_due  113418 non-null  int64         
 11  risk_a_clients    113418 non-null  int64         
 12  risk_a_balance    113418 non-null  float64       
 13  risk_b_clients    113418 non-null  int64         
 14  risk_b_balance 

### 3.3 Remove duplicated subtotals

- Some groups have a TOTAL row (code 5) plus details rows (codes 10 to 25). If we added all of them, the money would be counted twice.
- for example in Bancolombia's LIBRANZA for August 2026, the TOTAL row (code 5) and the OTROS row (code 15) had exactly the same numbers. the VIS and NO VIS groups only had a code 5 row because the had a single item.
- we only keep rows with code 5. It is the TOTAL when a group has details, or the product itself when the group has only one item.
- no TOTAL row has a code other than 5 (0 rows), and 40,651 TOTAL rows + 47,208 single-item rows = 87,859 rows.
- `df_clean` now has 87,859 rows and no double counting. The 24,743 detail rows are still safe in the raw schema.

In [20]:
df_clean[(df_clean["entity_name"].str.contains("BANCOLOMBIA", case=False)) & (df_clean["group_name"].str.contains("LIBRANZA", case=False)) & (df_clean["cutoff_date"] == df_clean["cutoff_date"].max())]

,entity_type,entity_code,entity_name,cutoff_date,group_code,group_name,product_code,product_name,total_balance,current_balance,clients_past_due,risk_a_clients,risk_a_balance,risk_b_clients,risk_b_balance,risk_c_clients,risk_c_balance,risk_d_clients,risk_d_balance,risk_e_clients,risk_e_balance
25327,1,7,Bancolombia,2026-09-30,4,LIBRANZA,5,LIBRANZA TOTAL,"6,014,272,754,540.28","5,850,208,680,020.18",7730,195291,"5,641,542,733,215.37",3712,"87,395,997,067.95",2318,"49,435,717,239.95",8654,"174,057,467,194.59",3089,"61,840,839,822.42"
25349,1,7,Bancolombia,2026-09-30,4,LIBRANZA,15,LIBRANZA OTROS,"6,014,272,754,540.28","5,850,208,680,020.18",7730,195291,"5,641,542,733,215.37",3712,"87,395,997,067.95",2318,"49,435,717,239.95",8654,"174,057,467,194.59",3089,"61,840,839,822.42"
55611,1,7,Bancolombia,2026-09-30,17,LIBRANZA VIVIENDA VIS,5,LIBRANZA VIVIENDA VIS TOTAL,"261,386,094.31","210,149,713.21",6,28,"210,149,713.21",1,"5,765,295.77",3,"22,861,242.57",2,"22,609,842.76",0,0.00
56595,1,7,Bancolombia,2026-09-30,18,LIBRANZA VIVIENDA NO VIS,5,LIBRANZA VIVIENDA NO VIS TOTAL,"639,901,324.12","639,901,324.12",0,6,"639,901,324.12",0,0.00,0,0.00,0,0.00,0,0.00


In [21]:
print(df_clean[(df_clean["product_name"].str.contains("TOTAL", case=False)) & (df_clean["product_code"] != "5")].shape[0])
df_clean[(df_clean["product_name"].str.contains("TOTAL", case=False)) & (df_clean["product_code"] == "5")].shape[0]

0


40944

In [22]:
df_clean = df_clean[df_clean["product_code"] == "5"]

In [23]:
df_clean[~df_clean["product_name"].str.contains("TOTAL", case=False)].shape

(47548, 21)

In [24]:
df_clean.shape

(88492, 21)

### 3.4 Group into credit types

- The Superintendencia Financiera classifies the credit into 4 categories:
    - **Consumer:** loans to individuals to buy consumer goods or pay for services.
    - **Commercial:** Loans to companies to finance their business activities.
    - **Microcredit:** Small loans to Microbusinesses, up to 120 SMMLV (monthly minimum wages).
    - **Housing:** For the acquisition, construction, repair or improvement of a home. 

- We classified each of the 32 groups by its `group_code` in a dictionary called `CREDIT_TYPES`. We used the code instead of `group_name` because names have accents and abbrevations, while codes are short and never change. Then `.map()` look up each code and writes its type in a new column called `credit_type`.
- Some groups were tricky. CARTERA COMERCIAL CONSTRUCCION sounds like housing, but it is a commercial loan to a construction company. CARTERA COMERCIAL MICROEMPRESA is commercial, not microcredit. And LEASING HABITACIONAL is housing, while LEASING FINANCIERO is commercial.
- In conclusion, `credit_type` has 0 empty values, and the 4 types add up to 87,859 rows. Commercial has the most rows because it has 10 groups, but more rows do not mean more money. 

In [25]:
df_clean[["group_code", "group_name"]].drop_duplicates().sort_values("group_name")

,group_code,group_name
86246,27,CARTERA COMERCIAL CONSTRUCCION
21527,19,CARTERA COMERCIAL CORPORATIVO
61707,20,CARTERA COMERCIAL EMPRESARIAL
79514,24,CARTERA COMERCIAL FACTOTING
81603,25,CARTERA COMERCIAL FINAN E INSTITUCIO
93195,28,CARTERA COMERCIAL LEASING FINANCIERO
21666,22,CARTERA COMERCIAL MICROEMPRESA
84063,26,CARTERA COMERCIAL MONEDA EXTRANJERA
73249,23,CARTERA COMERCIAL OFICIAL O GOBIERNO
66610,21,CARTERA COMERCIAL PYMES


In [26]:
CREDIT_TYPES = {
    "1": "consumer",
    "2": "consumer",
    "3": "consumer",
    "4": "consumer",
    "5": "consumer",
    "6": "consumer",
    "7": "microcredit",
    "8": "microcredit",
    "9": "housing",
    "10": "housing",
    "11": "housing",
    "12": "housing",
    "13": "housing",
    "14": "housing",
    "15": "housing",
    "16": "housing",
    "17": "housing",
    "18": "housing",
    "19": "commercial",
    "20": "commercial",
    "21": "commercial",
    "22": "commercial",
    "23": "commercial",
    "24": "commercial",
    "25": "commercial",
    "26": "commercial",
    "27": "commercial",
    "28": "commercial",
    "29": "consumer",
    "30": "consumer",
    "31": "housing",
    "32": "housing"
}

In [27]:
df_clean["credit_type"] = df_clean["group_code"].map(CREDIT_TYPES)

In [28]:
df_clean["credit_type"].isna().sum()

np.int64(0)

In [29]:
df_clean["credit_type"].value_counts()

credit_type
commercial     34266
consumer       27210
housing        20843
microcredit     6173
Name: count, dtype: int64

### 3.5 Calculate the past-due balance and delinquency rate

- The dashboard needs two numbers: how much money is past due and what percentage of the portfolio it represents.

#### 3.5.1 Past-due balance

- We calculate the past-due balance as the total balance minus the current balance: past_due = total_balance - current_balance.

In [30]:
df_clean["past_due"] = df_clean["total_balance"] - df_clean["current_balance"]

#### 3.5.2 Negative past-due values

- 27 rows had tiny negative values (a few cents, from rounding), so we set them to 0.
- 1 row was a source error: C.A. Credifinanciera (Feb 2016, microcredit) reported a current balance higher than the total. We removed it with a rule that drops past-due values below −1 peso, so it also catches future errors.
- 87858 rows remain.

In [31]:
df_clean[df_clean["past_due"] < -1][["entity_name", "cutoff_date", "group_name", "total_balance", "current_balance", "past_due"]]

,entity_name,cutoff_date,group_name,total_balance,current_balance,past_due
40390,C.A. Credifinanciera Cf,2016-02-29,MICROCREDITOS > 25 Y HASTA 120 SMMLV,"592,316,487.18","735,375,657.21","-143,059,170.03"


In [32]:
df_clean = df_clean[df_clean["past_due"] >= -1]

In [33]:
df_clean.shape # Check how many rows remain.

(88491, 23)

In [34]:
df_clean[df_clean["past_due"] < 0].head(3) # Show examples of negative cents (rounding)

,entity_type,entity_code,entity_name,cutoff_date,group_code,group_name,product_code,product_name,total_balance,current_balance,clients_past_due,risk_a_clients,risk_a_balance,risk_b_clients,risk_b_balance,risk_c_clients,risk_c_balance,risk_d_clients,risk_d_balance,risk_e_clients,risk_e_balance,credit_type,past_due
43461,4,118,Crezcamos S.A.,2024-07-31,9,VIVIENDA VIS PESOS,5,VIS PESOS TOTAL,"414,407,191.85","414,407,191.86",0,0,0.00,0,0.00,0,0.00,0,0.00,0,0.00,housing,-0.01
43469,4,118,Crezcamos S.A.,2024-09-30,9,VIVIENDA VIS PESOS,5,VIS PESOS TOTAL,"410,939,785.31","410,939,785.32",0,0,0.00,0,0.00,0,0.00,0,0.00,0,0.00,housing,-0.01
48158,1,55,Finandina Bic O Banco Finandina Bic,2025-12-31,11,VIVIENDA NO VIS PESOS,5,NO VIS PESOS TOTAL,"972,039,023.86","972,039,023.90",0,11,"972,039,023.90",0,0.00,0,0.00,0,0.00,0,0.00,housing,-0.04


In [35]:
(df_clean["past_due"] < 0).sum() # Count small negative values (cent rounding)

np.int64(28)

In [36]:
df_clean["past_due"] = df_clean["past_due"].clip(lower=0) # Set the small negative values to 0

In [37]:
(df_clean["past_due"] < 0).sum() # Check that no negative values are left

np.int64(0)

#### 3.5.3 Rows with a zero total balance

- 1198 rows had a total balance of 0, and we can't divide by 0 to get the deliquency rate.
- 84 of them had money in the risk columns (e.g., Banco Caja Social, 2015), but it was only 0.00014% of the portfolio, so we dropped all 1,198 rows.
- 86,660 rows remain

In [38]:
(df_clean["total_balance"] == 0).sum()

np.int64(1207)

In [39]:
df_clean[df_clean["total_balance"] == 0][["total_balance", "current_balance", "past_due", "clients_past_due"]].sum()

total_balance        0.00
current_balance      0.00
past_due             0.00
clients_past_due   373.00
dtype: float64

In [40]:
df_clean[(df_clean["total_balance"] == 0) & (df_clean["clients_past_due"] > 0)].head(3)

,entity_type,entity_code,entity_name,cutoff_date,group_code,group_name,product_code,product_name,total_balance,current_balance,clients_past_due,risk_a_clients,risk_a_balance,risk_b_clients,risk_b_balance,risk_c_clients,risk_c_balance,risk_d_clients,risk_d_balance,risk_e_clients,risk_e_balance,credit_type,past_due
55409,1,23,Banco De Occidente,2022-04-30,17,LIBRANZA VIVIENDA VIS,5,LIBRANZA VIVIENDA VIS TOTAL,0.00,0.00,19,0,0.00,0,0.00,0,0.00,0,0.00,0,0.00,housing,0.00
56385,1,23,Banco De Occidente,2022-04-30,18,LIBRANZA VIVIENDA NO VIS,5,LIBRANZA VIVIENDA NO VIS TOTAL,0.00,0.00,57,0,0.00,0,0.00,0,0.00,0,0.00,0,0.00,housing,0.00
104243,1,30,Banco Caja Social,2015-01-31,30,CREDITOS DE CONSUMO PARA EMPLEADOS,5,CRÉDITOS DE CONSUMO PARA EMPLEADOS,0.00,0.00,19,946,"2,992,867,376.11",6,"25,750,577.93",5,"23,095,236.37",9,"22,049,161.55",13,"58,975,169.71",consumer,0.00


In [41]:
RISK_BALANCE_COLUMNS = ["risk_a_balance", "risk_b_balance", "risk_c_balance", "risk_d_balance", "risk_e_balance"] # Balance columns by risk category (A-E)

In [42]:
df_clean[(df_clean["total_balance"] == 0) & (df_clean["clients_past_due"] > 0)][RISK_BALANCE_COLUMNS].sum(axis=1).sum() # Sum the risk balances of zero-total rows with clients past due

np.float64(34532058014.79001)

In [43]:
df_clean[df_clean["total_balance"] == 0][RISK_BALANCE_COLUMNS].sum(axis=1).sum()  # Sum the risk balances of all zero-total rows

np.float64(109479572290.14)

In [44]:
(df_clean[df_clean["total_balance"] == 0][RISK_BALANCE_COLUMNS].sum(axis=1) > 0).sum() # Calculate how many rows there are where the sum of the row is higher than 0

np.int64(84)

In [45]:
(df_clean[df_clean["total_balance"] == 0][RISK_BALANCE_COLUMNS].sum(axis=1).sum() / df_clean["total_balance"].sum()) * 100 # Compare the hidden balance with the total portfolio (%) 

np.float64(0.00013929865652815548)

In [46]:
df_clean = df_clean[df_clean["total_balance"] > 0] # Remove the rows with a total of 0, as they account for only 0.00014% of the portfolio.

In [47]:
df_clean.shape

(87284, 23)

#### 3.5.4 Delinquency rate

- We calculated the delinquency rate (npl_ratio) by dividing the past-due balance by the total balance. All values are between 0 and 1.
- The system delinquency rate is 4.84% (sum/sum), but the mean per row is 8%, because in a simple mean a tiny product weighs the same as a huge one. That’s why the dashboard will always use sum/sum.
- 865 rows are 100% past due; almost all are tiny products (about 163,678 million pesos in total).
- Some rows with a 100% rate have no clients past due, which doesn’t add up. We look into them in section 3.6.

In [48]:
df_clean["npl_ratio"] = df_clean["past_due"] / df_clean["total_balance"] # Calculate the delinquency rate

In [49]:
df_clean.describe() # Check that the rate is between 0 and 1

,cutoff_date,total_balance,current_balance,clients_past_due,risk_a_clients,risk_a_balance,risk_b_clients,risk_b_balance,risk_c_clients,risk_c_balance,risk_d_clients,risk_d_balance,risk_e_clients,risk_e_balance,past_due,npl_ratio
count,87284,"87,284.00","87,284.00","87,284.00","87,284.00","87,284.00","87,284.00","87,284.00","87,284.00","87,284.00","87,284.00","87,284.00","87,284.00","87,284.00","87,284.00","87,284.00"
mean,2020-11-22 13:48:13.561248,"900,433,249,760.69","856,939,727,015.14","2,337.48","28,477.53","807,167,718,303.19","1,017.38","24,701,461,514.96",954.52,"16,352,196,146.85",896.09,"19,222,091,960.70","1,012.86","18,586,343,944.79","43,493,522,745.55",0.08
min,2015-01-31 00:00:00,7.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
25%,2017-11-30 00:00:00,"6,894,851,431.78","6,199,320,335.33",0.00,20.00,"2,861,902,182.12",0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,"24,250,281.00",0.00
50%,2020-10-31 00:00:00,"75,574,796,210.16","70,145,505,061.87",19.00,360.00,"56,210,014,618.74",11.00,"790,740,370.66",5.00,"290,225,976.71",6.00,"356,746,626.00",4.00,"172,979,085.00","2,011,872,526.86",0.04
75%,2023-10-31 00:00:00,"573,029,527,044.87","543,477,875,840.50",441.00,"4,961.50","478,248,192,778.95",194.00,"11,577,487,148.13",92.00,"6,114,570,529.75",127.00,"6,484,852,674.77",125.00,"6,357,695,047.35","23,949,490,310.14",0.09
max,2026-09-30 00:00:00,"55,275,439,979,726.90","54,576,855,957,540.70","191,385.00","1,660,900.00","53,183,439,925,496.50","154,695.00","1,731,142,468,080.21","947,270.00","1,710,085,771,111.43","2,449,842.00","2,350,074,437,521.43","164,677.00","2,092,576,448,087.10","3,199,945,482,117.00",1.00
std,NaN,"2,459,299,094,430.65","2,365,183,266,826.94","10,061.15","119,338.16","2,260,901,962,779.64","4,845.87","75,626,265,142.44","14,430.62","59,264,121,063.57","9,358.06","72,229,142,431.56","5,596.31","78,990,341,865.86","126,075,537,197.13",0.14


In [50]:
(df_clean["past_due"].sum()/df_clean["total_balance"].sum())*100 # Calculate the system-wide delinquency rate, weighted (4.84%)

np.float64(4.830288392516687)

In [51]:
df_clean[df_clean["npl_ratio"] == 1].shape # Count the rows that are 100% past due

(872, 24)

In [52]:
df_clean[df_clean["npl_ratio"] == 1]["total_balance"].sum() # Sum the total balance of those rows

np.float64(164086002777.9)

### 3.6 Check the consistency of the past-due balance

- 69 rows were 100% past due but had 0 customers in past due, so the question was wheter it was real delinquency or a mistake.
- We checked the past due columns ("vencida") by age in the raw data. If they have money, the delinquency is real; if they are 0, the row doesn't add up.
- For 64 rows the delinquency was real: the money was in the past due columns, and only the client and risk columns were wrong.
- 5 rows were reporting errors: no past-due money and no clients past due. they come from 4 entities, are all consumer loans with everything in risk category A, mostly from 2015, and add up to about 1692 million pesos.
- We dropped these 5 rows because we don't know the correct value and don't want to make them up.
- 86,655 rows remain

In [53]:
suspicious = df_clean[(df_clean["npl_ratio"] == 1) & (df_clean["clients_past_due"] == 0)] # set aside the rows that are 100% past due but have no clients past due. 

In [54]:
suspicious.shape

(69, 24)

In [55]:
ARREARS_COLUMNS = [c for c in df_raw.columns if "vencida" in c] # List of the past due columns by age (from raw)

In [56]:
suspicious_arrears = df_raw.loc[suspicious.index, ARREARS_COLUMNS].astype(float).sum(axis=1) # Sum the past due money of each suspicious row.

In [57]:
print((suspicious_arrears > 0).sum())
print((suspicious_arrears == 0).sum())

64
5


In [58]:
report_errors = suspicious[suspicious_arrears == 0] # Save the rows with 100% delinquency but no overdue amounts (reporting errors)

In [59]:
df_clean = df_clean.drop(report_errors.index) # Drop the report errors

In [60]:
report_errors["total_balance"].sum()

np.float64(1692148894.96)

In [61]:
report_errors

,entity_type,entity_code,entity_name,cutoff_date,group_code,group_name,product_code,product_name,total_balance,current_balance,clients_past_due,risk_a_clients,risk_a_balance,risk_b_clients,risk_b_balance,risk_c_clients,risk_c_balance,risk_d_clients,risk_d_balance,risk_e_clients,risk_e_balance,credit_type,past_due,npl_ratio
32375,1,62,Mibanco S.A.,2017-08-31,6,OTROS PORTAFOLIOS DE CONSUMO,5,CONSUMO OTROS TOTAL,"29,343.33",0.00,0,1,"29,343.33",0,0.00,0,0.00,0,0.00,0,0.00,consumer,"29,343.33",1.00
34499,4,59,Leasing Corficolombiana,2015-10-31,6,OTROS PORTAFOLIOS DE CONSUMO,5,CONSUMO OTROS TOTAL,"690,227,988.00",0.00,0,16,"690,227,988.00",0,0.00,0,0.00,0,0.00,0,0.00,consumer,"690,227,988.00",1.00
34504,4,59,Leasing Corficolombiana,2015-11-30,6,OTROS PORTAFOLIOS DE CONSUMO,5,CONSUMO OTROS TOTAL,"660,845,005.00",0.00,0,16,"660,845,005.00",0,0.00,0,0.00,0,0.00,0,0.00,consumer,"660,845,005.00",1.00
104276,1,56,Banco Falabella S.A.,2015-03-31,30,CREDITOS DE CONSUMO PARA EMPLEADOS,5,CRÉDITOS DE CONSUMO PARA EMPLEADOS,"162,142,376.43",0.00,0,46,"162,142,376.43",0,0.00,0,0.00,0,0.00,0,0.00,consumer,"162,142,376.43",1.00
106835,4,65,Leasing Bolivar,2015-01-31,30,CREDITOS DE CONSUMO PARA EMPLEADOS,5,CRÉDITOS DE CONSUMO PARA EMPLEADOS,"178,904,182.20",0.00,0,58,"178,904,182.20",0,0.00,0,0.00,0,0.00,0,0.00,consumer,"178,904,182.20",1.00


## Part 4 - Check the clean data

### 4.1 Check for nulls and duplicates

- There are no missing values and no duplicates: each entity, month and group appears only once.

In [ ]:
df_clean.isna().sum().sum() # Check that there are no missing values.

np.int64(0)

In [ ]:
df_clean.duplicated(subset=["entity_type", "entity_code", "cutoff_date", "group_code"]).sum() # Check that each entity, month and group appears only once.

np.int64(0)

### 4.2 Check the date range by credit type

- All 4 credit types start in January 2015 and end in September 2026, so the dashboard lines will cover tha same period.

In [ ]:
df_clean.groupby("credit_type")["cutoff_date"].agg(["min", "max"]) # Check the first and last date for each credit type

,min,max
credit_type,,
commercial,2015-01-31,2026-09-30
consumer,2015-01-31,2026-09-30
housing,2015-01-31,2026-09-30
microcredit,2015-01-31,2026-09-30


### 4.3 Check the delinquency rate by credit type

- We calculated the delinquency rate by credit type for the last month (September 2026). The order makes sense: microcredit (6.62%) and housing (6.59) are the highest, followed by consumer (4.88%) and commercial (3.06%). The high housing rate is worth exploring in the dashboard.

In [116]:
last_month = df_clean[df_clean["cutoff_date"] == df_clean["cutoff_date"].max()]

In [117]:
by_type = last_month.groupby("credit_type")[["past_due", "total_balance"]].sum()

In [133]:
by_type["npl_ratio"] = (by_type["past_due"]/by_type["total_balance"]*100).round(2)

In [134]:
by_type

,past_due,total_balance,npl_ratio
credit_type,,,
commercial,"12,766,920,585,104.33","417,580,509,807,601.69",3.06
consumer,"11,199,410,472,898.57","229,676,809,328,407.16",4.88
housing,"9,845,470,207,299.84","149,363,199,224,796.28",6.59
microcredit,"1,735,157,702,637.75","26,225,032,520,387.06",6.62
